In [1]:
import pandas as pd

from granuscore.loader import JSONLineReader

reader = JSONLineReader()

In [12]:
NAME_LOOKUP = {
    'hit-random_anchors': 'HiT Dist0 + RandomAnch',
    'hit-random': 'HiT Dist0 + Random',
    'hit-radial_anchors': 'HiT Dist0 + RadialAnch',
    'hit-nearest_neighbor': 'HiT Dist0 + NN',
    'hit-embedding-random_anchors': 'HiT RandomAnch',
    'hit-embedding-random': 'HiT Random',
    'hit-embedding-radial_anchors': 'HiT RadialAnch',
    'hit-embedding-nearest_neighbor': 'HiT NN',
    'all-min-random_anchors': 'MiniLM RandomAnch',
    'all-min-random': 'MiniLM Random',
    'all-min-nearest_neighbor': 'MiniLM NN',
    'llm-gpt-4-1-mini': 'GPT-4.1 mini',
    'llm-gpt-4-1': 'GPT-4.1',
    'len': 'Word Count',
    'hit-dist0': 'HiT Dist0',
    'wordnet': 'WordNet'
}

stats = []

data = reader.read('min_evaluate_lgb_models.jsonl')
test_stats_dist0 = data[0]['test_answers']['stats_dist0']
del test_stats_dist0['avg_granuscores']
del test_stats_dist0['intra_significance']
stats.append({
        'name': NAME_LOOKUP.get('hit-dist0'),
        **test_stats_dist0
    })

for d in data:
    test_stats = d['test_answers']['stats']
    del test_stats['avg_granuscores']
    del test_stats['intra_significance']

    name = d['name']
    if 'hit-random_anchors' in name and not '999' in name:
        continue

    for k, v in NAME_LOOKUP.items():
        if k in name:
            name = v
            break

    stats.append({
        'name': name,
        **test_stats
    })
stats = pd.DataFrame(stats)

In [13]:
stats

,name,not_nan_percentage,global_pairwise_acc,global_kendall_tau,global_pearson,avg_intra_pairwise_acc,avg_intra_kendall_tau,mean_intra_full_correct
0,HiT Dist0,100.000000,0.8082,0.5070,0.5237,0.8786,0.7573,0.7350
1,HiT Dist0 + RandomAnch,100.000000,0.8376,0.5553,0.6969,0.8903,0.7806,0.7436
2,HiT Dist0 + RadialAnch,100.000000,0.8322,0.5466,0.6891,0.8883,0.7766,0.7385
3,HiT Dist0 + Random,100.000000,0.8201,0.5251,0.6667,0.8682,0.7365,0.7085
4,HiT Dist0 + NN,100.000000,0.8286,0.5404,0.6755,0.8702,0.7405,0.7154
5,MiniLM RandomAnch,100.000000,0.6745,0.2873,0.3776,0.7115,0.4231,0.4915
6,MiniLM Random,100.000000,0.6755,0.2879,0.3787,0.7057,0.4114,0.4880
7,MiniLM NN,100.000000,0.6740,0.2865,0.3694,0.6950,0.3900,0.4897
8,Word Count,100.000000,0.5049,0.0415,0.0107,0.5154,0.0791,0.2880
9,GPT-4.1 mini,100.000000,0.7676,0.6984,0.7404,0.8158,0.8824,0.5821


In [14]:
order = [
    "Word Count",
    "WordNet",
    "GPT-4.1 mini",
    "HiT Dist0",
    "MiniLM NN",
    "MiniLM Random",
    "MiniLM RandomAnch",
    "HiT NN",
    "HiT Random",
    "HiT RadialAnch",
    "HiT RandomAnch",
    "HiT Dist0 + NN",
    "HiT Dist0 + Random",
    "HiT Dist0 + RadialAnch",
    "HiT Dist0 + RandomAnch",
]

main_stats = stats.set_index("name").loc[order].reset_index()
main_stats = main_stats[[
    "name",
    "global_pairwise_acc",
    "avg_intra_pairwise_acc",
    "mean_intra_full_correct",
]]


main_stats = main_stats.rename(columns={
    "name": "Method",
    "global_pairwise_acc": "\makecell{PW\\Acc.}",
    "global_kendall_tau": "Kendall $\\tau$",
    "global_pearson": "Pearson $r$",
    "avg_intra_pairwise_acc": "\makecell{Intra PW\\Acc.}",
    "avg_intra_kendall_tau": "Intra Kendall $\\tau$",
    "mean_intra_full_correct": "Exact",
})

num_cols = main_stats.columns.drop("Method")
main_stats[num_cols] = (main_stats[num_cols] * 100).round(2)

# copy for formatting
main_stats_fmt = main_stats.copy()

for col in num_cols:
    # get indices of largest and second largest
    order = main_stats[col].sort_values(ascending=False).index
    best = order[0]
    second = order[1]

    for i in main_stats.index:
        val = main_stats.loc[i, col]

        if i == best:
            main_stats_fmt.loc[i, col] = f"\\textbf{{{val:.2f}}}"
        elif i == second:
            main_stats_fmt.loc[i, col] = f"\\textit{{{val:.2f}}}"
        else:
            main_stats_fmt.loc[i, col] = f"{val:.2f}"

# export latex
latex = main_stats_fmt.to_latex(index=False, escape=False)

print(latex)

\begin{tabular}{llll}
\toprule
Method & \makecell{PW\Acc.} & \makecell{Intra PW\Acc.} & Exact \\
\midrule
Word Count & 50.49 & 51.54 & 28.80 \\
WordNet & 58.12 & 67.32 & 60.39 \\
GPT-4.1 mini & 76.76 & 81.58 & 58.21 \\
HiT Dist0 & 80.82 & 87.86 & 73.50 \\
MiniLM NN & 67.40 & 69.50 & 48.97 \\
MiniLM Random & 67.55 & 70.57 & 48.80 \\
MiniLM RandomAnch & 67.45 & 71.15 & 49.15 \\
HiT NN & 81.79 & 86.47 & 70.17 \\
HiT Random & 80.80 & 84.74 & 69.32 \\
HiT RadialAnch & 82.31 & 88.35 & 71.71 \\
HiT RandomAnch & 83.00 & 88.15 & 72.48 \\
HiT Dist0 + NN & 82.86 & 87.02 & 71.54 \\
HiT Dist0 + Random & 82.01 & 86.82 & 70.85 \\
HiT Dist0 + RadialAnch & \textit{83.22} & \textit{88.83} & \textit{73.85} \\
HiT Dist0 + RandomAnch & \textbf{83.76} & \textbf{89.03} & \textbf{74.36} \\
\bottomrule
\end{tabular}



<>:30: SyntaxWarning: invalid escape sequence '\m'
<>:33: SyntaxWarning: invalid escape sequence '\m'
<>:30: SyntaxWarning: invalid escape sequence '\m'
<>:33: SyntaxWarning: invalid escape sequence '\m'
/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_95125/3640000939.py:30: SyntaxWarning: invalid escape sequence '\m'
  "global_pairwise_acc": "\makecell{PW\\Acc.}",
/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_95125/3640000939.py:33: SyntaxWarning: invalid escape sequence '\m'
  "avg_intra_pairwise_acc": "\makecell{Intra PW\\Acc.}",
/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_95125/3640000939.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '50.49' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  main_stats_fmt.loc[i, col] = f"{val:.2f}"
/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_95125/3640000939.py:58: Fu

In [16]:
order = [
    "Word Count",
    "WordNet",
    "GPT-4.1 mini",
    "HiT Dist0",
    "MiniLM NN",
    "MiniLM Random",
    "MiniLM RandomAnch",
    "HiT NN",
    "HiT Random",
    "HiT RadialAnch",
    "HiT RandomAnch",
    "HiT Dist0 + NN",
    "HiT Dist0 + Random",
    "HiT Dist0 + RadialAnch",
    "HiT Dist0 + RandomAnch",
]

main_stats = stats.set_index("name").loc[order].reset_index()
main_stats = main_stats[[
    "name",
    "global_kendall_tau",
    "global_pearson",
    "avg_intra_kendall_tau",
]]


main_stats = main_stats.rename(columns={
    "name": "Method",
    "global_pairwise_acc": "\makecell{PW\\Acc.}",
    "global_kendall_tau": "Kendall $\\tau$",
    "global_pearson": "Pearson $r$",
    "avg_intra_pairwise_acc": "\makecell{Intra PW\\Acc.}",
    "avg_intra_kendall_tau": "Intra Kendall $\\tau$",
    "mean_intra_full_correct": "Exact",
})

num_cols = main_stats.columns.drop("Method")
main_stats[num_cols] = (main_stats[num_cols] * 100).round(2)

# copy for formatting
main_stats_fmt = main_stats.copy()

for col in num_cols:
    # get indices of largest and second largest
    order = main_stats[col].sort_values(ascending=False).index
    best = order[0]
    second = order[1]

    for i in main_stats.index:
        val = main_stats.loc[i, col]

        if i == best:
            main_stats_fmt.loc[i, col] = f"\\textbf{{{val:.2f}}}"
        elif i == second:
            main_stats_fmt.loc[i, col] = f"\\textit{{{val:.2f}}}"
        else:
            main_stats_fmt.loc[i, col] = f"{val:.2f}"

# export latex
latex = main_stats_fmt.to_latex(index=False, escape=False)

print(latex)

\begin{tabular}{llll}
\toprule
Method & Kendall $\tau$ & Pearson $r$ & Intra Kendall $\tau$ \\
\midrule
Word Count & 4.15 & 1.07 & 7.91 \\
WordNet & 25.54 & 28.15 & 61.15 \\
GPT-4.1 mini & \textbf{69.84} & \textbf{74.04} & \textbf{88.24} \\
HiT Dist0 & 50.70 & 52.37 & 75.73 \\
MiniLM NN & 28.65 & 36.94 & 39.00 \\
MiniLM Random & 28.79 & 37.87 & 41.14 \\
MiniLM RandomAnch & 28.73 & 37.76 & 42.31 \\
HiT NN & 52.28 & 65.13 & 72.93 \\
HiT Random & 50.53 & 64.41 & 69.49 \\
HiT RadialAnch & 53.16 & 66.04 & 76.70 \\
HiT RandomAnch & 54.28 & 67.58 & 76.30 \\
HiT Dist0 + NN & 54.04 & 67.55 & 74.05 \\
HiT Dist0 + Random & 52.51 & 66.67 & 73.65 \\
HiT Dist0 + RadialAnch & 54.66 & 68.91 & 77.66 \\
HiT Dist0 + RandomAnch & \textit{55.53} & \textit{69.69} & \textit{78.06} \\
\bottomrule
\end{tabular}



<>:30: SyntaxWarning: invalid escape sequence '\m'
<>:33: SyntaxWarning: invalid escape sequence '\m'
<>:30: SyntaxWarning: invalid escape sequence '\m'
<>:33: SyntaxWarning: invalid escape sequence '\m'
/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_95125/3936947089.py:30: SyntaxWarning: invalid escape sequence '\m'
  "global_pairwise_acc": "\makecell{PW\\Acc.}",
/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_95125/3936947089.py:33: SyntaxWarning: invalid escape sequence '\m'
  "avg_intra_pairwise_acc": "\makecell{Intra PW\\Acc.}",
/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_95125/3936947089.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4.15' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  main_stats_fmt.loc[i, col] = f"{val:.2f}"
/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_95125/3936947089.py:58: Fut

### K Ablation

In [47]:
NAME_LOOKUP = {
    'hit-random_anchors': 'HiT RandomAnch {k}',
}

stats = []

data = reader.read('min_evaluate_lgb_models.jsonl')

for d in data:
    test_stats = d['test_answers']['stats']
    del test_stats['avg_granuscores']
    del test_stats['intra_significance']

    name = d['name']
    if not 'hit-random_anchors' in name:
        continue

    for k, v in NAME_LOOKUP.items():
        if k in name:
            name = v.format(k=name.split('-')[-1])
            break

    stats.append({
        'name': name,
        **test_stats
    })
stats = pd.DataFrame(stats)

In [48]:
stats

,name,global_pairwise_acc,global_kendall_tau,global_pearson,avg_intra_pairwise_acc,avg_intra_kendall_tau,mean_intra_full_correct
0,HiT RandomAnch 33,0.8337,0.5490,0.6874,0.8993,0.7990,0.7361
1,HiT RandomAnch 66,0.8316,0.5455,0.6762,0.8853,0.7707,0.7098
2,HiT RandomAnch 99,0.8313,0.5450,0.6741,0.8865,0.7729,0.7123
3,HiT RandomAnch 333,0.8330,0.5478,0.6784,0.8670,0.7339,0.7033
4,HiT RandomAnch 666,0.8244,0.5338,0.6732,0.8826,0.7652,0.7074
5,HiT RandomAnch 999,0.8376,0.5553,0.6969,0.8903,0.7806,0.7131
6,HiT RandomAnch 1332,0.8247,0.5340,0.6670,0.8715,0.7430,0.6943
7,HiT RandomAnch 1665,0.8345,0.5502,0.6918,0.8900,0.7801,0.7189


In [50]:
main_stats = stats.set_index("name").reset_index()
main_stats = main_stats[[
    "name",
    "global_pairwise_acc",
]]


main_stats = main_stats.rename(columns={
    "name": "Method",
    "global_pairwise_acc": "PW Acc.",
})

num_cols = main_stats.columns.drop("Method")
main_stats[num_cols] = (main_stats[num_cols] * 100).round(2)

# copy for formatting
main_stats_fmt = main_stats.copy()

for col in num_cols:
    # get indices of largest and second largest
    order = main_stats[col].sort_values(ascending=False).index
    best = order[0]
    second = order[1]

    for i in stats.index:
        val = main_stats.loc[i, col]

        if i == best:
            main_stats_fmt.loc[i, col] = f"\\textbf{{{val:.2f}}}"
        elif i == second:
            main_stats_fmt.loc[i, col] = f"\\textit{{{val:.2f}}}"
        else:
            main_stats_fmt.loc[i, col] = f"{val:.2f}"

# export latex
latex = main_stats_fmt.to_latex(index=False, escape=False)

print(latex)

\begin{tabular}{ll}
\toprule
Method & PW Acc. \\
\midrule
HiT RandomAnch 33 & 83.37 \\
HiT RandomAnch 66 & 83.16 \\
HiT RandomAnch 99 & 83.13 \\
HiT RandomAnch 333 & 83.30 \\
HiT RandomAnch 666 & 82.44 \\
HiT RandomAnch 999 & \textbf{83.76} \\
HiT RandomAnch 1332 & 82.47 \\
HiT RandomAnch 1665 & \textit{83.45} \\
\bottomrule
\end{tabular}



/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_46387/3456682833.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '83.37' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  main_stats_fmt.loc[i, col] = f"{val:.2f}"
